# VadCLIP Temporal CNN: kernel 5, residual off
Chi tat skip connection trong hai CNN block. Giu scoring_head, Top-K, loss va lich learning rate cua luot kernel 5 truoc. Train from scratch voi seed 234 trong 15 epoch; danh gia epoch 10 va 15 cua cung mot luot.

In [ ]:
import json
import shutil
import subprocess
import sys
from pathlib import Path

import torch
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/Finetune VadCLIP')
SRC_DIR = PROJECT_ROOT / 'VadCLIP' / 'src'
LIST_DIR = PROJECT_ROOT / 'VadCLIP' / 'list'
FEATURE_ROOT = Path('/content/UCFClipFeatures')
OUTPUT_ROOT = PROJECT_ROOT / 'code' / 'results'
RUN_TAG = 'cnn_k5_nores_s234_e15'
VARIANT = 'k5_head_topk_nores'
SEED = 234
FINAL_EPOCH = 15
EVAL_EPOCHS = (10, 15)

if not torch.cuda.is_available():
    raise RuntimeError('Can chon GPU runtime tren Colab.')
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(PROJECT_ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ftfy', 'regex', 'scipy', 'scikit-learn', 'pandas'], check=True)
if not FEATURE_ROOT.is_dir():
    archives = [PROJECT_ROOT / name for name in ('UCFClipFeatures.zip', 'UCFClipFeatures.tar', 'UCFClipFeatures.tar.gz')]
    archive = next((path for path in archives if path.is_file()), None)
    if archive is None:
        raise FileNotFoundError('Can UCFClipFeatures tai /content hoac archive tren Drive.')
    local_archive = Path('/content') / archive.name
    if not local_archive.is_file() or local_archive.stat().st_size != archive.stat().st_size:
        shutil.copy2(archive, local_archive)
    command = ['unzip', '-q', str(local_archive), '-d', '/content'] if archive.suffix == '.zip' else ['tar', '-xf', str(local_archive), '-C', '/content']
    subprocess.run(command, check=True)
for path in (SRC_DIR / 'model_cnn_vadclip.py', SRC_DIR / 'ucf_train_cnn_vadclip.py',
             SRC_DIR / 'ucf_evaluate_cnn_vadclip.py', SRC_DIR / 'ucf_cnn_common.py',
             LIST_DIR / 'ucf_CLIP_rgb_relative.csv', LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv',
             LIST_DIR / 'gt_ucf.npy', LIST_DIR / 'gt_segment_ucf.npy', LIST_DIR / 'gt_label_ucf.npy',
             FEATURE_ROOT):
    if not path.exists():
        raise FileNotFoundError(path)
print('GPU:', torch.cuda.get_device_name(0))
print('Feature root:', FEATURE_ROOT)
print('Output tag:', RUN_TAG)


## Smoke test
Kiem tra skip connection da tat, score va gradient huu han; khong train model day du trong cell nay.

In [ ]:
sys.path.insert(0, str(SRC_DIR))
from model_cnn_vadclip import TemporalCNNClassifier

cnn = TemporalCNNClassifier(kernel_size=5, score_mode='scoring_head', use_residual=False)
assert all(not block.use_residual for block in cnn.blocks)
features = torch.randn(2, 20, 512)
lengths = torch.tensor([13, 20])
scores = cnn(features, lengths)
assert scores.shape == (2, 20, 1) and torch.isfinite(scores).all()
assert torch.equal(scores[0, 13:], torch.zeros_like(scores[0, 13:]))
scores[:, :13].sum().backward()
assert cnn.blocks[0].conv1.weight.grad is not None
print('Smoke test PASS: kernel=5, residual=off, shape=', tuple(scores.shape))
del cnn


## Train va evaluate
Chi train mot variant moi den epoch 15. Danh gia checkpoint epoch 10 va 15 rieng biet; model va ket qua 10 epoch cu khong bi ghi de.

In [ ]:
def run_command(command):
    print('Running:', ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(command, cwd=SRC_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait() != 0:
        raise RuntimeError('Child process failed; traceback is printed above.')

checkpoint_dir = SRC_DIR / 'model' / 'epoch_checkpoints_cnn_vadclip' / RUN_TAG / VARIANT
checkpoint = checkpoint_dir / f'epoch_{FINAL_EPOCH:03d}.pth'
train_dir = OUTPUT_ROOT / 'train' / RUN_TAG / VARIANT
eval_dir = OUTPUT_ROOT / 'evaluate' / RUN_TAG / VARIANT
analyze_dir = OUTPUT_ROOT / 'analyze' / RUN_TAG / VARIANT
config_path = train_dir / 'config.json'

if checkpoint.is_file():
    if not config_path.is_file():
        raise RuntimeError('Checkpoint exists without config; use another RUN_TAG.')
    config = json.loads(config_path.read_text(encoding='utf-8'))
    expected = {'kernel_size': 5, 'score_mode': 'scoring_head', 'residual_mode': 'off',
                'aggregation': 'topk', 'lambda_alignment': 1.0, 'lambda_text': 1.0,
                'seed': SEED, 'batch_size': 64, 'lr': 1e-5, 'max_epoch': FINAL_EPOCH,
                'scheduler_milestones': [4, 8], 'scheduler_rate': 0.1,
                'visual_length': 256, 'feature_root': str(FEATURE_ROOT),
                'train_list': str(LIST_DIR / 'ucf_CLIP_rgb_relative.csv')}
    if any(config.get(key) != value for key, value in expected.items()):
        raise RuntimeError('Existing checkpoint config does not match; use another RUN_TAG.')
    print('Matching final checkpoint exists; skip training:', checkpoint)
else:
    train_cmd = [sys.executable, '-u', str(SRC_DIR / 'ucf_train_cnn_vadclip.py'),
                 '--feature-root', str(FEATURE_ROOT),
                 '--train-list', str(LIST_DIR / 'ucf_CLIP_rgb_relative.csv'),
                 '--kernel-size', '5', '--score-mode', 'scoring_head',
                 '--residual-mode', 'off', '--aggregation', 'topk',
                 '--lambda-alignment', '1.0', '--lambda-text', '1.0',
                 '--seed', str(SEED), '--batch-size', '64', '--lr', '1e-5',
                 '--max-epoch', str(FINAL_EPOCH), '--scheduler-milestones', '4', '8',
                 '--scheduler-rate', '0.1', '--num-workers', '2',
                 '--visual-length', '256', '--epoch-checkpoint-dir', str(checkpoint_dir),
                 '--output-model-path', str(SRC_DIR / 'model' / 'cnn_vadclip' / RUN_TAG / f'{VARIANT}.pth'),
                 '--results-dir', str(train_dir)]
    run_command(train_cmd)

for epoch in EVAL_EPOCHS:
    epoch_checkpoint = checkpoint_dir / f'epoch_{epoch:03d}.pth'
    if not epoch_checkpoint.is_file():
        raise FileNotFoundError(epoch_checkpoint)
    epoch_eval_dir = eval_dir / f'epoch_{epoch:03d}'
    epoch_analyze_dir = analyze_dir / f'epoch_{epoch:03d}'
    metrics_path = epoch_eval_dir / 'metrics.json'
    if metrics_path.is_file():
        if json.loads(metrics_path.read_text(encoding='utf-8')).get('model_path') != str(epoch_checkpoint):
            raise RuntimeError('Cached metrics point to a different checkpoint; use another RUN_TAG.')
        print('Matching evaluation exists; skip evaluation:', metrics_path)
        continue
    eval_cmd = [sys.executable, '-u', str(SRC_DIR / 'ucf_evaluate_cnn_vadclip.py'),
                '--model-type', 'cnn', '--model-path', str(epoch_checkpoint),
                '--feature-root', str(FEATURE_ROOT),
                '--test-list', str(LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv'),
                '--gt-path', str(LIST_DIR / 'gt_ucf.npy'),
                '--gt-segment-path', str(LIST_DIR / 'gt_segment_ucf.npy'),
                '--gt-label-path', str(LIST_DIR / 'gt_label_ucf.npy'),
                '--kernel-size', '5', '--score-mode', 'scoring_head',
                '--residual-mode', 'off', '--train-config-json', str(config_path),
                '--evaluate-dir', str(epoch_eval_dir), '--analyze-dir', str(epoch_analyze_dir)]
    run_command(eval_cmd)
print('Train result:', train_dir)
print('Evaluate result:', eval_dir)
print('Analyze result:', analyze_dir)


## So sanh epoch 10 va 15
So sanh hai checkpoint trong cung mot luot 15 epoch va doi chieu cac luot kernel 5 truoc neu metrics con tren Drive. Khong chon epoch theo AUC test.

In [ ]:
keys = ('classifier_auc', 'classifier_ap', 'alignment_auc', 'alignment_ap', 'ano_auc', 'avg_mAP')
print('Model | AUC1 | AP1 | AUC2 | AP2 | Ano-AUC | mAP')
for tag in ('cnn_vadclip_v1', 'cnn_vadclip_seed234_repeat'):
    old_path = OUTPUT_ROOT / 'evaluate' / tag / 'k5_head_topk' / 'metrics.json'
    if not old_path.is_file():
        print('Missing reference:', old_path)
        continue
    old = json.loads(old_path.read_text(encoding='utf-8'))
    print(tag + ' residual on | ' + ' | '.join(f'{old[key]:.2f}' for key in keys))
old_off_path = OUTPUT_ROOT / 'evaluate' / 'cnn_k5_nores_s234' / VARIANT / 'metrics.json'
if old_off_path.is_file():
    old_off = json.loads(old_off_path.read_text(encoding='utf-8'))
    print('residual off, old 10 epoch | ' + ' | '.join(f'{old_off[key]:.2f}' for key in keys))
metrics_by_epoch = {}
for epoch in EVAL_EPOCHS:
    path = eval_dir / f'epoch_{epoch:03d}' / 'metrics.json'
    metrics_by_epoch[epoch] = json.loads(path.read_text(encoding='utf-8'))
    print(f'residual off, new epoch {epoch} | ' + ' | '.join(f'{metrics_by_epoch[epoch][key]:.2f}' for key in keys))
    summary_path = analyze_dir / f'epoch_{epoch:03d}' / 'summary.json'
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print(f"  Top-15 anomaly fraction (macro): {summary['selected_anomaly_fraction_abnormal_video_macro']:.3f}")
first_epoch, last_epoch = EVAL_EPOCHS[0], EVAL_EPOCHS[-1]
delta = {key: metrics_by_epoch[last_epoch][key] - metrics_by_epoch[first_epoch][key] for key in keys}
print(f'Delta epoch {last_epoch} - epoch {first_epoch} | ' + ' | '.join(f'{delta[key]:+.2f}' for key in keys))
print('Final checkpoint:', checkpoint)
